# Lesson 02 - মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক অন্বেষণ

**Microsoft Agent Framework (MAF)** হলো AI এজেন্ট তৈরি করার জন্য একটি একীভূত ফ্রেমওয়ার্ক। এটি একটি পরিষ্কার, গঠনযোগ্য আর্কিটেকচার সরবরাহ করে যার চারটি মূল নির্মাণ ব্লক রয়েছে:

- **Client** – একটি AI মডেল এন্ডপয়েন্টের সাথে সংযোগ স্থাপন করে এবং যোগাযোগ পরিচালনা করে
- **Agent** – ক্লায়েন্টকে নির্দেশাবলী এবং টুল সংজ্ঞার সাথে মোড়কে রাখে
- **Tools** – কাস্টম ফাংশন দ্বারা এজেন্টের ক্ষমতা বৃদ্ধি করে যা মডেল 호출 করতে পারে
- **Session** – বহুতর কথোপকথনের জন্য সংলাপের ইতিহাস বজায় রাখে

এই পাঠে, আমরা একটি **ট্রাভেল বুকিং এজেন্ট** তৈরি করব যা এই ধারণাগুলি ব্যবহার করে গন্তব্য উপলভ্যতা পরীক্ষা করবে।


## সেটআপ


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## এজেন্ট ফ্রেমওয়ার্ক আর্কিটেকচার বোঝা

মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক একটি স্তরযুক্ত আর্কিটেকচার অনুসরণ করে:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **ক্লায়েন্ট** – একটি `AzureAIProjectAgentProvider` একটি Azure OpenAI ডিপ্লয়মেন্টের সাথে সংযুক্ত হয়। এটি প্রমাণীকরণ, অনুরোধ ফরম্যাটিং, এবং প্রতিক্রিয়া পার্সিং পরিচালনা করে।
2. **এজেন্ট** – ক্লায়েন্ট থেকে `provider.create_agent()` এর মাধ্যমে তৈরি, এজেন্ট মডেল অ্যাক্সেসকে নির্দেশনা (সিস্টেম প্রম্পট) এবং সরঞ্জামের সাথে মিলিত করে।
3. **সরঞ্জাম** – `@tool` দিয়ে সজ্জিত পাইথন ফাংশনসমূহ যা এজেন্ট কর্ম সম্পাদন বা ডেটা পুনরুদ্ধারের জন্য আহ্বান করতে পারে।
4. **সেশন** – একটি `AgentSession` অবজেক্ট (agent.create_session() এর মাধ্যমে তৈরি) যা কথোপকথনের ইতিহাস সংরক্ষণ করে, বহুতল সংলাপ সক্ষম করে যেখানে এজেন্ট পূর্বের প্রসঙ্গ স্মরণ করে।

চলুন প্রতিটি স্তর এক ধাপে এক ধাপে তৈরি করি।


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool ডেকোরেটর দিয়ে টুল যোগ করা

টুলগুলি এজেন্টদের টেক্সট তৈরি ছাড়াও কার্যক্রম গ্রহণ করতে দেয়। `@tool` ডেকোরেটর একটি নিয়মিত পাইথন ফাংশনকে এমন কিছুতে রূপান্তর করে যা এজেন্ট কল করতে পারে।

মূল বিষয়গুলি:
- মডেল প্রতিটি প্যারামিটার বুঝতে পারে তা নিশ্চিত করার জন্য `Annotated[type, "description"]` ব্যবহার করুন।
- ডকস্ট্রিং টুলের বর্ণনা হয়ে দাঁড়ায় যা মডেল দেখবে।
- `approval_mode="never_require"` মানে টুল স্বয়ংক্রিয়ভাবে ব্যবহারকারীর নিশ্চিতকরণ ছাড়াই চালানো হবে।


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## টুলস সহ একটি এজেন্ট তৈরি করা

এখন আমরা ক্লায়েন্ট, নির্দেশাবলী, এবং টুলসকে একটি এজেন্টে মিলিত করি। `instructions` সিস্টেম প্রম্পট হিসাবে কাজ করে — এটি এজেন্টের ব্যক্তিত্ব এবং আচরণ নির্ধারণ করে।


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## সেশন সহ মাল্টি-টার্ন কনভারসেশন

একটি `AgentSession` (যা `agent.create_session()` এর মাধ্যমে তৈরি করা হয়) একটি কথোপকথনের সমস্ত বার্তা ট্র্যাক করে। একই সেশন প্রতিটি `agent.run()` কল-এ পাঠানোর মাধ্যমে, এজেন্ট সম্পূর্ণ কথোপকথনের ইতিহাস দেখতে পায় এবং পূর্বের বার্তাগুলিতে ফিরে যেতে পারে।

আমরা `tools=[check_destination_availability]` প্রদান করি যাতে এজেন্ট প্রতিটি টার্নে আমাদের অ্যাভেইলেবিলিটি চেকার কল করতে পারে।


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## সারাংশ

এই পাঠে আপনি Microsoft Agent Framework-এর চারটি মূল স্তম্ভ অন্বেষণ করেছেন:

| ধারণা | আপনি যা শিখেছেন |
|---------|------------------|
| **ক্লায়েন্ট** | `AzureAIProjectAgentProvider` ক্রেডেনশিয়াল-ভিত্তিক অথের মাধ্যমে Azure OpenAI-র সাথে সংযুক্ত হয় |
| **এজেন্ট** | `provider.create_agent()` একটি মডেল সংযোগকে নির্দেশাবলী এবং একটি নামের সাথে প্যাকেজ করে |
| **টুলস** | `@tool` ডেকোরেটর Python ফাংশনগুলি উন্মুক্ত করে যাতে এজেন্ট সেগুলি কল করতে পারে |
| **সেশন** | `agent.create_session()` বহু টার্ন জুড়ে কথোপকথনের ইতিহাস বজায় রাখে |

এই নির্মাণ ব্লকগুলি একসাথে মিলিত হয়ে এজেন্ট তৈরি করে যা সুমধুর স্বাভাবিক কথোপকথন বজায় রাখতে পারে, বাহ্যিক ফাংশন কল করতে পারে, এবং প্রসঙ্গ বজায় রাখতে পারে — যা পরবর্তী পাঠে অধিক উন্নত এজেন্টিক প্যাটার্নের ভিত্তি রূপে কাজ করে।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**দায়ত্ব স্বীকার**:  
এই নথিটি AI অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনুবাদ করা হয়েছে। আমরা যথাসাধ্য সঠিকতার চেষ্টা করি, তবে স্বয়ংক্রিয় অনুবাদে ভুল বা অসূচ্চতা থাকতে পারে। আসল নথিটি তার নিজস্ব ভাষায় সবচেয়ে নির্ভরযোগ্য উৎস হিসেবে বিবেচনা করা উচিত। গুরুত্বপূর্ণ তথ্যের জন্য, পেশাদার মানব অনুবাদ পরামর্শ দেওয়া হয়। এই অনুবাদের ব্যবহারে সৃষ্ট কোনও ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়ী নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
